<table style="width: 100%; border-collapse: collapse; border: none; background: #fffbeb; border-left: 6px solid #f59e0b; border-radius: 8px; padding: 20px; box-shadow: 0 2px 4px rgba(0,0,0,0.05);">
  <tr style="border: none;">
    <td style="vertical-align: middle; border: none; padding: 15px 20px;">
      <h1 style="margin: 0; color: #78350f; font-size: 2em; font-family: system-ui, -apple-system, sans-serif; font-weight: 800; letter-spacing: -0.02em;">
        💡 Candados, Huellas y un Libro a Prueba de Borrones: Cómo se Protegen los Datos 🔐
      </h1>
      <p style="margin: 6px 0 0 0; color: #b45309; font-size: 1.15em; font-weight: 600; font-family: system-ui, -apple-system, sans-serif;">
        Especialización en Ciencia de Datos | Big Data
      </p>
      <p style="margin: 4px 0 0 0; color: #92400e; font-size: 0.95em; font-family: system-ui, -apple-system, sans-serif;">
        Universidad Santo Tomás — Seccional Tunja
      </p>
    </td>
    <td style="text-align: right; vertical-align: middle; border: none; padding: 15px 20px; width: 30%;">
      <span style="background: #f59e0b; color: #ffffff; padding: 6px 14px; border-radius: 20px; font-size: 0.85em; font-weight: 700; display: inline-block; margin-bottom: 8px;">
        💡 Para Dummies • Módulo 04
      </span><br>
      <span style="color: #78350f; font-size: 0.85em;">Docente: Santiago A. Zúñiga M.</span><br>
      <a href="mailto:gestorvirtualcienciadatos@ustatunja.edu.co" style="color: #b45309; font-size: 0.8em; text-decoration: none; font-weight: 500;">gestorvirtualcienciadatos@ustatunja.edu.co</a>
    </td>
  </tr>
</table>

<div align="center" style="margin-top: 15px; margin-bottom: 15px;">
  <a href="https://colab.research.google.com/github/sazuniga06/Data-Science-Programming---USTA-Tunja-Repository/blob/main/Big%20Data/04%20-%20Seguridad%20y%20Gobernanza%20de%20Big%20Data/Para%20Dummies/01_Tecnicas_Seguridad_Dummies.ipynb" target="_parent">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" style="vertical-align: middle;"/>
  </a>
</div>

---
## ¿Qué vamos a aprender aquí? 🎈

Este cuaderno es la versión **«para no ingenieros»** del [cuaderno 01 del Módulo 04 (Técnicas de seguridad)](../01_Tecnicas_de_Seguridad_Encriptacion_Acceso_Auditoria.ipynb). Después de saber **qué amenazas hay** (cuaderno anterior), toca **defenderse**. Pensemos en una **caja fuerte**, un **sello de cera** y el **libro de portería**:

| En la vida diaria | En seguridad de datos |
|---|---|
| **Un mensaje secreto** con un código que solo tú y tu amigo conocen | **Cifrado**: sin la clave, el mensaje no se entiende |
| **Un sello de cera** en la carta: si está roto, alguien la abrió | **Huella digital** (*hash*): si el documento cambia, la huella cambia |
| **No guardar tu clave**, sino solo cómo se ve «su huella» | **Contraseñas con sal**: guardar la huella, no la clave |
| **Un libro de portería** donde cada renglón menciona al anterior: si arrancan una hoja, **se nota** | **Bitácora inviolable** (cadena de huellas) |

*(Este cuaderno usa solo Python básico. El «código secreto» que implementamos es un **juguete** para entender la idea: **jamás** se usa para proteger algo real; para eso existen candados **de verdad**, ya probados por expertos y disponibles en bibliotecas como `cryptography`.)* Al terminar podrás explicar:
1. Qué es **cifrar**, y por qué un código **fácil de adivinar** (como el del colegio) **no** protege nada.
2. Qué es una **huella digital** y cómo sirve para **detectar cambios** y para **no guardar contraseñas**.
3. Cómo un **libro de portería encadenado** hace que **arrancar una hoja se note**.

> 🔗 Si luego quieres ver la versión **con candados de verdad (cifrado por sobre, contraseñas con función lenta, control de acceso y privacidad diferencial)**: [cuaderno estándar](../01_Tecnicas_de_Seguridad_Encriptacion_Acceso_Auditoria.ipynb). Antes de esto: [La tienda de Doña Rosa](00_Amenazas_Seguridad_Dummies.ipynb).

---
## 1. El Código Secreto del Colegio: y por qué NO sirve 🔒

De niños jugábamos a **correr cada letra tres puestos** en el abecedario: la **A** pasa a ser **D**, la **B** pasa a ser **E**… Eso se llama **cifrado César** (por Julio César, que lo usaba). El **«tres»** es la **clave**: quien la conoce, descifra. Lo implementamos como **juguete** para ver **cómo funciona** y, sobre todo, **por qué es débil**.

In [ ]:
import hashlib
import hmac
import secrets
import string

ALFABETO = string.ascii_lowercase                                   # 26 letras (juguete: sin tildes ni eñes)

def cesar(texto, clave):
    """JUGUETE didactico: corre cada letra `clave` puestos. NUNCA usar para proteger datos reales."""
    salida = []
    for ch in texto.lower():
        salida.append(ALFABETO[(ALFABETO.index(ch) + clave) % 26] if ch in ALFABETO else ch)
    return "".join(salida)

secreto = cesar("nos vemos en el parque de tunja a las cinco", 3)
print("Mensaje cifrado:", secreto)
print("Con la clave correcta:", cesar(secreto, -3))
assert cesar(secreto, -3) == "nos vemos en el parque de tunja a las cinco" and secreto != "nos vemos en el parque de tunja a las cinco"

# ¿Por que es debil? Solo hay 26 claves posibles: se prueban TODAS en un instante
intentos = {k: cesar(secreto, -k) for k in range(1, 26)}
legibles = [k for k, t in intentos.items() if "parque" in t]
print(f"\nProbando las 25 claves posibles, la única que da un texto con sentido es la {legibles[0]}:", intentos[legibles[0]])
assert legibles == [3] and len(intentos) == 25

---
**Lección:** el mensaje se ve ilegible, pero **solo hay 25 claves posibles** y un computador las prueba **todas en un instante**: en cuanto una da un texto con sentido, se acabó el secreto. Un **candado de verdad** funciona con la misma idea (una clave que mezcla el mensaje) pero con **tantas claves posibles** que **ni probando sin parar con todos los computadores del mundo** alcanzaría una vida entera *(orden de magnitud, aproximado, verificar)*. Además, esos candados **detectan** si alguien cambió el mensaje cifrado a mitad de camino. Moraleja: **la seguridad no está en que el truco sea secreto**, está en que **la clave sea imposible de adivinar**, y por eso **nunca se inventa un candado propio**: se usa uno ya probado por expertos.

---

## 2. La Huella Digital: el Sello de Cera de los Datos 🖐️

Una **huella digital** (*hash*) es un número de aspecto aleatorio que se calcula a partir de **un texto**: **el mismo texto siempre da la misma huella**, pero **cambiar una sola coma** la cambia **por completo**, y **desde la huella no se puede recuperar el texto**. Tiene dos usos que vemos con dos ejemplos: **detectar que algo cambió** y **no guardar contraseñas**.

In [ ]:
def huella(texto):
    return hashlib.sha256(texto.encode()).hexdigest()[:16]          # solo los primeros 16 caracteres, para leer facil

nota_1 = "Doña Rosa le fía 50 mil pesos a Pedro."
nota_2 = "Doña Rosa le fía 500 mil pesos a Pedro."                  # alguien agrego un cero
print("Huella de la nota original :", huella(nota_1))
print("Huella de la nota alterada :", huella(nota_2))
assert huella(nota_1) == huella(nota_1) and huella(nota_1) != huella(nota_2)
print("-> Un cero de más y la huella es totalmente distinta: así se descubre que la nota cambió.")

# CONTRASEÑAS: no se guarda la clave, se guarda su huella con una "sal" (un numero al azar propio de cada persona)
def guardar(clave):
    sal = secrets.token_hex(4)                                      # sal aleatoria de cada persona
    return sal, hashlib.sha256((sal + clave).encode()).hexdigest()[:16]

def revisar(clave, guardado):
    sal, h = guardado
    return hmac.compare_digest(hashlib.sha256((sal + clave).encode()).hexdigest()[:16], h)

ana, beto = guardar("tunja2025"), guardar("tunja2025")             # Ana y Beto escogen LA MISMA clave
print("\nLo que guarda la tienda de Ana :", ana)
print("Lo que guarda la tienda de Beto:", beto)
assert ana != beto and revisar("tunja2025", ana) and not revisar("tunja2026", ana)

# Sin sal, la misma clave daria la misma huella y se vería quien repite clave
print("Sin sal, la huella de 'tunja2025' sería para los dos:", huella("tunja2025"), "-> se notaría que repiten clave")

---
**Lección:** un **cero de más** y la huella queda **irreconocible**: por eso se usa para **detectar cambios** en un documento o en una base de datos. Y para las **contraseñas**: la tienda **no guarda tu clave**, guarda **su huella mezclada con una sal** (un número al azar solo tuyo). Así, aunque **Ana y Beto usen la misma clave**, **lo guardado es distinto**, y quien robe la lista **no ve la clave** ni sabe **quién repite** la suya. Cuando alguien entra, se calcula la huella con la sal de esa persona y se **compara**. *(Una advertencia que se ve en el [cuaderno estándar](../01_Tecnicas_de_Seguridad_Encriptacion_Acceso_Auditoria.ipynb): con una huella **rápida** como esta, un ladrón que robe la lista igual puede **probar claves comunes**; por eso los sistemas reales usan huellas **lentas a propósito**.)*

---

## 3. El Libro de Portería a Prueba de Borrones 📒

El portero anota **quién entró y a qué hora**. Un administrador deshonesto podría **cambiar un renglón** («yo nunca estuve aquí»). Para que **se note**, cada renglón guarda **la huella del renglón anterior**: si alguien **altera** un renglón, **todas las huellas siguientes** dejan de cuadrar, como una **cadena** en la que un eslabón roto se ve. Lo armamos con las mismas herramientas de arriba.

In [ ]:
def armar_libro(renglones):
    """Cada renglon guarda su texto, la huella del anterior y su propia huella (que incluye la del anterior)."""
    libro, anterior = [], "inicio"
    for texto in renglones:
        h = huella(anterior + "|" + texto)
        libro.append({"texto": texto, "anterior": anterior, "huella": h})
        anterior = h
    return libro

def revisar_libro(libro):
    """Devuelve None si todo cuadra, o el numero del primer renglon que no cuadra."""
    anterior = "inicio"
    for i, r in enumerate(libro):
        if r["anterior"] != anterior or r["huella"] != huella(anterior + "|" + r["texto"]):
            return i
        anterior = r["huella"]
    return None

libro = armar_libro(["8:00 entra Marta", "8:15 entra Pedro", "9:30 entra el administrador", "10:00 sale Marta", "11:20 sale el administrador"])
assert revisar_libro(libro) is None
print("Libro original: todo cuadra ->", revisar_libro(libro))

import copy
alterado = copy.deepcopy(libro)
alterado[2]["texto"] = "9:30 entra Luz"                              # el administrador borra su rastro
print("Tras cambiar el renglón 2 (cuenta desde 0), el primer renglón que no cuadra es el:", revisar_libro(alterado))
assert revisar_libro(alterado) == 2

arrancada = copy.deepcopy(libro); del arrancada[1]                     # alguien arranca una hoja
print("Tras arrancar el renglón 1, el primer renglón que no cuadra es el:", revisar_libro(arrancada))
assert revisar_libro(arrancada) == 1

---
**Lección:** cambiar un renglón o **arrancar una hoja** **rompe la cadena** en ese punto, y el portero (o un auditor) lo descubre **con solo recalcular las huellas**. Pero tiene un **límite honesto**: si el administrador es **muy hábil** y **recalcula todas las huellas** que siguen, la cadena **vuelve a cuadrar**, o si **corta las últimas hojas**, nadie lo nota; por eso se guarda **la última huella en otro lugar** (en la notaría, en otro computador que él no controle): ese «ancla» delata cualquier reescritura. *(Es la misma idea de las **cadenas de bloques**, que veremos en el Módulo 05; el [cuaderno estándar](../01_Tecnicas_de_Seguridad_Encriptacion_Acceso_Auditoria.ipynb) lo prueba con un ancla y con alertas automáticas.)*

---

## Resumen en una frase 🎯

> **Proteger datos es cifrarlos con un candado ya probado (nunca inventado), usar huellas digitales para detectar cambios y para guardar contraseñas con sal, y llevar un libro encadenado donde arrancar una hoja se nota.**

### Lo que aprendiste hoy:
- ✅ **Cifrar** es mezclar el mensaje con una clave; un código con **pocas claves posibles** (como el César) se rompe probándolas todas.
- ✅ **Huella digital:** el mismo texto da la misma huella; **una coma de diferencia** cambia todo; sirve para detectar cambios.
- ✅ **Contraseñas:** se guarda la huella **con sal**, no la clave; dos personas con la misma clave quedan con registros distintos.
- ✅ **Libro encadenado:** alterar o arrancar un renglón **se nota**; para que no lo reescriban sin que se note, se guarda la última huella **fuera** del libro.

> 🎓 **Siguiente paso:** el [cuaderno estándar](../01_Tecnicas_de_Seguridad_Encriptacion_Acceso_Auditoria.ipynb) usa los **candados de verdad**, rota claves, controla accesos y **publica estadísticas sin delatar a nadie**. Para cerrar el módulo: [El inventario del pueblo y quién responde por cada cosa](02_Gobernanza_Datos_Dummies.ipynb).

---
##### 🛠️ Práctica 1: Encuentra el renglón alterado en un libro nuevo

Usa `armar_libro`, `revisar_libro` y `copy` de arriba:

1. Arma `libro_2 = armar_libro([...])` con **cuatro renglones** de tu invención (por ejemplo, entradas y salidas de la tienda).
2. Crea `copia = copy.deepcopy(libro_2)` y cambia el texto del **renglón 3** (el último) por otro.
3. Calcula `primero_malo = revisar_libro(copia)` y comprueba con `assert` que es `3`.
4. Comprueba también que `revisar_libro(libro_2)` es `None` (el libro original sigue cuadrando).

In [ ]:
# Escribe tu código aquí

# libro_2 = armar_libro([...])
# copia = copy.deepcopy(libro_2)
# copia[3]["texto"] = ...
# primero_malo = revisar_libro(copia)


<details>
<summary><b>💡 Haz clic aquí para ver la solución guiada...</b></summary>

```python
libro_2 = armar_libro(["7:00 abre Doña Rosa", "7:30 llega el pan", "12:00 cierra para almorzar", "14:00 abre de nuevo"])
copia = copy.deepcopy(libro_2)
copia[3]["texto"] = "14:00 no abrió"
primero_malo = revisar_libro(copia)
print("El primer renglón que no cuadra es el:", primero_malo, "| el libro original:", revisar_libro(libro_2))
assert primero_malo == 3 and revisar_libro(libro_2) is None
print("Al alterar un renglón, ese ya no cuadra con su huella; si arrancas uno, el siguiente deja de apuntar al anterior. Para ocultarlo habría que recalcular todas las huellas que siguen.")
```
</details>

---


---
### 🧠 ¿Ya lo tienes claro?

1. ¿Por qué el código César **no** sirve para proteger datos reales aunque el mensaje «se vea» ilegible? ¿Qué tiene un candado de verdad que el César no?
2. Dos personas escogen la misma contraseña. ¿Qué se **guarda** de cada una y por qué no son iguales? ¿Qué se vería si no se usara sal?
3. El administrador cambia un renglón del libro y vuelve a calcular **todas** las huellas siguientes: el libro **cuadra** otra vez. ¿Cómo lo descubrirías con **una sola huella guardada afuera**?

---
<div align="center">
  <p style="font-size: 0.9em; color: #78350f;">
    © 2026 <b>Universidad Santo Tomás — Seccional Tunja</b><br>
    <i>Especialización en Ciencia de Datos | Big Data (Edición Didáctica Para Dummies)</i>
  </p>
</div>